## Import libs

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.python.data import Dataset
from tensorflow.keras.optimizers import Adam
import seaborn as sns


from falsb4mpa.modeling.zhang.learning.multi_adv import train_loop as zhang_train
from falsb4mpa.dataset.load_data import load_data
from falsb4mpa.evaluation.evaluation import compute_predictive_metrics, compute_fair_metrics, compute_adv_metrics, compute_tradeoff, fair_evaluation, compute_intersectional_fair_metrics
from falsb4mpa.modeling.zhang.models.multi_adv import ZhangMultAdv

## Preliminaries

In [2]:
batch_size = 64
epochs = 100
learning_rate = 0.001

In [3]:
# cv_seeds = [13]
cv_seeds = [13, 29, 42, 55, 73]

## Load data

In [4]:
data_name = 'compas-mpa-bin-wout-agg'

In [5]:
x, y, a1, a2 = load_data(data_name)
raw_data = (x, y, a1, a2)

In [6]:
xdim = x.shape[1]
ydim = y.shape[1]
a1dim = a1.shape[1]
a2dim = a2.shape[1]
zdim = 8
print(xdim, ydim, a1dim, a2dim, zdim)

11 1 1 1 8


## Result file

In [7]:
header = [
    "model_name", "cv_seed", 
    "clas_acc", "f1-micro", "f1-macro",
    "a1_dp", "a1_deqodds", "a1_deqopp", 
    "a1_TN_g0", "a1_FP_g0", "a1_FN_g0", "a1_TP_g0", "a1_TN_g1", "a1_FP_g1", "a1_FN_g1", "a1_TP_g1", 
    "a2_dp", "a2_deqodds", "a2_deqopp", 
    "a2_TN_g0", "a2_FP_g0", "a2_FN_g0", "a2_TP_g0", "a2_TN_g1", "a2_FP_g1", "a2_FN_g1", "a2_TP_g1",
    "wc_spd", "wc_aod", "wc_eod",
    "last_cosine_similarity"
]

results = []

## Testing

### DemPar

In [8]:
fairdef = "DemPar"

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test = train_test_split(
        x, y, a1, a2, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a1_train, a2_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a1_test, a2_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    opt = Adam(learning_rate=learning_rate)

    model = ZhangMultAdv(xdim=xdim, ydim=ydim, a1dim=a1dim, a2dim=a2dim, batch_size=batch_size, fairdef=fairdef)
    
    ret, dULa1, dULa2, cos_sim = zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A1, A2, Y_hat, A1_hat, A2_hat = fair_evaluation(model, test_data)
    
    clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix = compute_predictive_metrics(Y, Y_hat)
    
    adv1_acc = compute_adv_metrics(A1, A1_hat)
    adv2_acc = compute_adv_metrics(A2, A2_hat)
    
    a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1 = compute_fair_metrics(Y, A1, Y_hat, a1dim)
    a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1 = compute_fair_metrics(Y, A2, Y_hat, a2dim)

    wc_spd, wc_aod, wc_eod = compute_intersectional_fair_metrics(Y, A1, A2, Y_hat, a1dim, a2dim)


    # fair_metrics = (dp, deqodds, deqopp)
    # tradeoff = []
    # for fair_metric in fair_metrics:
    #     tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    # result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    result = ['MultAdvBin4DP', cv_seed]
    result += [clas_acc, clas_f1_micro, clas_f1_macro]
    result += [a1_dp, a1_deqodds, a1_deqopp] + a1_metrics_g0 + a1_metrics_g1 
    result += [a2_dp, a2_deqodds, a2_deqopp] + a2_metrics_g0 + a2_metrics_g1
    result += [wc_spd, wc_aod, wc_eod]
    result += [cos_sim]


    results.append(result)

    del(opt, x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test, train_data, test_data, model, ret)
    del(Y, A1, A2, Y_hat, A1_hat, A2_hat)
    del(clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix, adv1_acc, adv2_acc)
    del(a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1, a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1)
    del(wc_spd, wc_aod, wc_eod)
    del(cos_sim)

2026-05-15 12:45:59.774593: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 1 | Clf loss/acc 0.83/0.28 | Adv1 loss/acc 0.56/0.81 | Adv2 loss/acc 0.94/0.34 | Cos Sim -0.13


2026-05-15 12:46:02.292232: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 2 | Clf loss/acc 0.74/0.33 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 0.91/0.34 | Cos Sim -0.14
> Epoch: 3 | Clf loss/acc 0.69/0.46 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 0.88/0.34 | Cos Sim -0.16


2026-05-15 12:46:07.211142: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


...

> Epoch: 64 | Clf loss/acc 0.67/0.75 | Adv1 loss/acc 0.98/0.52 | Adv2 loss/acc 0.66/0.71 | Cos Sim 0.26
> Epoch: 65 | Clf loss/acc 0.67/0.75 | Adv1 loss/acc 0.98/0.52 | Adv2 loss/acc 0.67/0.71 | Cos Sim 0.26
> Epoch: 66 | Clf loss/acc 0.68/0.75 | Adv1 loss/acc 0.99/0.52 | Adv2 loss/acc 0.67/0.71 | Cos Sim 0.26
> Epoch: 67 | Clf loss/acc 0.68/0.75 | Adv1 loss/acc 0.99/0.52 | Adv2 loss/acc 0.67/0.71 | Cos Sim 0.26
> Epoch: 68 | Clf loss/acc 0.68/0.75 | Adv1 loss/acc 1.00/0.52 | Adv2 loss/acc 0.67/0.71 | Cos Sim 0.26
> Epoch: 69 | Clf loss/acc 0.68/0.75 | Adv1 loss/acc 1.00/0.52 | Adv2 loss/acc 0.67/0.71 | Cos Sim 0.26
> Epoch: 70 | Clf loss/acc 0.68/0.75 | Adv1 loss/acc 1.00/0.52 | Adv2 loss/acc 0.67/0.71 | Cos Sim 0.26
> Epoch: 71 | Clf loss/acc 0.68/0.75 | Adv1 loss/acc 1.01/0.52 | Adv2 loss/acc 0.67/0.71 | Cos Sim 0.26
> Epoch: 72 | Clf loss/acc 0.69/0.75 | Adv1 loss/acc 1.01/0.52 | Adv2 loss/acc 0.67/0.71 | Cos Sim 0.27
> Epoch: 73 | Clf loss/acc 0.69/0.75 | Adv1 loss/acc 1.01/0.52 |

2026-05-15 12:51:08.913666: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 27 | Clf loss/acc 0.46/0.75 | Adv1 loss/acc 0.65/0.53 | Adv2 loss/acc 0.60/0.70 | Cos Sim 0.11
> Epoch: 28 | Clf loss/acc 0.46/0.75 | Adv1 loss/acc 0.65/0.51 | Adv2 loss/acc 0.60/0.69 | Cos Sim 0.12
> Epoch: 29 | Clf loss/acc 0.45/0.75 | Adv1 loss/acc 0.66/0.50 | Adv2 loss/acc 0.60/0.69 | Cos Sim 0.13
> Epoch: 30 | Clf loss/acc 0.45/0.75 | Adv1 loss/acc 0.66/0.48 | Adv2 loss/acc 0.60/0.69 | Cos Sim 0.14
> Epoch: 31 | Clf loss/acc 0.45/0.75 | Adv1 loss/acc 0.67/0.47 | Adv2 loss/acc 0.61/0.69 | Cos Sim 0.15
> Epoch: 32 | Clf loss/acc 0.45/0.76 | Adv1 loss/acc 0.67/0.47 | Adv2 loss/acc 0.61/0.69 | Cos Sim 0.15
> Epoch: 33 | Clf loss/acc 0.44/0.76 | Adv1 loss/acc 0.67/0.46 | Adv2 loss/acc 0.62/0.68 | Cos Sim 0.16
> Epoch: 34 | Clf loss/acc 0.44/0.76 | Adv1 loss/acc 0.67/0.46 | Adv2 loss/acc 0.62/0.68 | Cos Sim 0.16
> Epoch: 35 | Clf loss/acc 0.44/0.76 | Adv1 loss/acc 0.67/0.46 | Adv2 loss/acc 0.63/0.68 | Cos Sim 0.16
> Epoch: 36 | Clf loss/acc 0.44/0.76 | Adv1 loss/acc 0.67/0.46 |

2026-05-15 12:56:17.713346: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 54 | Clf loss/acc 0.53/0.75 | Adv1 loss/acc 0.86/0.48 | Adv2 loss/acc 0.64/0.69 | Cos Sim 0.22
> Epoch: 55 | Clf loss/acc 0.54/0.75 | Adv1 loss/acc 0.87/0.48 | Adv2 loss/acc 0.64/0.69 | Cos Sim 0.22
> Epoch: 56 | Clf loss/acc 0.54/0.75 | Adv1 loss/acc 0.87/0.48 | Adv2 loss/acc 0.64/0.69 | Cos Sim 0.22
> Epoch: 57 | Clf loss/acc 0.54/0.75 | Adv1 loss/acc 0.88/0.48 | Adv2 loss/acc 0.64/0.69 | Cos Sim 0.22
> Epoch: 58 | Clf loss/acc 0.54/0.75 | Adv1 loss/acc 0.88/0.48 | Adv2 loss/acc 0.65/0.69 | Cos Sim 0.22
> Epoch: 59 | Clf loss/acc 0.54/0.75 | Adv1 loss/acc 0.89/0.48 | Adv2 loss/acc 0.65/0.69 | Cos Sim 0.23
> Epoch: 60 | Clf loss/acc 0.54/0.75 | Adv1 loss/acc 0.89/0.48 | Adv2 loss/acc 0.65/0.69 | Cos Sim 0.23
> Epoch: 61 | Clf loss/acc 0.54/0.75 | Adv1 loss/acc 0.90/0.48 | Adv2 loss/acc 0.65/0.69 | Cos Sim 0.23
> Epoch: 62 | Clf loss/acc 0.54/0.75 | Adv1 loss/acc 0.90/0.48 | Adv2 loss/acc 0.65/0.69 | Cos Sim 0.23
> Epoch: 63 | Clf loss/acc 0.54/0.75 | Adv1 loss/acc 0.91/0.48 |

### EqOdds

In [9]:
fairdef = "EqOdds"

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test = train_test_split(
        x, y, a1, a2, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a1_train, a2_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a1_test, a2_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    opt = Adam(learning_rate=learning_rate)

    model = ZhangMultAdv(xdim=xdim, ydim=ydim, a1dim=a1dim, a2dim=a2dim, batch_size=batch_size, fairdef=fairdef)
    
    ret, dULa1, dULa2, cos_sim = zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A1, A2, Y_hat, A1_hat, A2_hat = fair_evaluation(model, test_data)
    
    clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix = compute_predictive_metrics(Y, Y_hat)
    
    adv1_acc = compute_adv_metrics(A1, A1_hat)
    adv2_acc = compute_adv_metrics(A2, A2_hat)
    
    a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1 = compute_fair_metrics(Y, A1, Y_hat, a1dim)
    a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1 = compute_fair_metrics(Y, A2, Y_hat, a2dim)

    wc_spd, wc_aod, wc_eod = compute_intersectional_fair_metrics(Y, A1, A2, Y_hat, a1dim, a2dim)


    # fair_metrics = (dp, deqodds, deqopp)
    # tradeoff = []
    # for fair_metric in fair_metrics:
    #     tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    # result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    result = ['MultAdvBin4EqOdds', cv_seed]
    result += [clas_acc, clas_f1_micro, clas_f1_macro]
    result += [a1_dp, a1_deqodds, a1_deqopp] + a1_metrics_g0 + a1_metrics_g1 
    result += [a2_dp, a2_deqodds, a2_deqopp] + a2_metrics_g0 + a2_metrics_g1
    result += [wc_spd, wc_aod, wc_eod]
    result += [cos_sim]


    results.append(result)

    del(opt, x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test, train_data, test_data, model, ret)
    del(Y, A1, A2, Y_hat, A1_hat, A2_hat)
    del(clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix, adv1_acc, adv2_acc)
    del(a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1, a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1)
    del(wc_spd, wc_aod, wc_eod)
    del(cos_sim)

> Epoch: 1 | Clf loss/acc 0.83/0.28 | Adv1 loss/acc 0.56/0.81 | Adv2 loss/acc 0.92/0.34 | Cos Sim -0.12
> Epoch: 2 | Clf loss/acc 0.74/0.32 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 0.87/0.34 | Cos Sim -0.13
> Epoch: 3 | Clf loss/acc 0.69/0.45 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 0.84/0.34 | Cos Sim -0.15
> Epoch: 4 | Clf loss/acc 0.66/0.57 | Adv1 loss/acc 0.58/0.81 | Adv2 loss/acc 0.81/0.34 | Cos Sim -0.16
> Epoch: 5 | Clf loss/acc 0.64/0.60 | Adv1 loss/acc 0.58/0.81 | Adv2 loss/acc 0.78/0.34 | Cos Sim -0.16
> Epoch: 6 | Clf loss/acc 0.62/0.65 | Adv1 loss/acc 0.58/0.81 | Adv2 loss/acc 0.75/0.36 | Cos Sim -0.16


2026-05-15 13:06:42.319674: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 7 | Clf loss/acc 0.62/0.68 | Adv1 loss/acc 0.59/0.81 | Adv2 loss/acc 0.73/0.40 | Cos Sim -0.15
> Epoch: 8 | Clf loss/acc 0.61/0.70 | Adv1 loss/acc 0.59/0.81 | Adv2 loss/acc 0.70/0.44 | Cos Sim -0.15
> Epoch: 9 | Clf loss/acc 0.60/0.71 | Adv1 loss/acc 0.59/0.81 | Adv2 loss/acc 0.68/0.54 | Cos Sim -0.13
> Epoch: 10 | Clf loss/acc 0.60/0.71 | Adv1 loss/acc 0.60/0.81 | Adv2 loss/acc 0.66/0.67 | Cos Sim -0.12
> Epoch: 11 | Clf loss/acc 0.60/0.71 | Adv1 loss/acc 0.60/0.81 | Adv2 loss/acc 0.64/0.76 | Cos Sim -0.10
> Epoch: 12 | Clf loss/acc 0.60/0.72 | Adv1 loss/acc 0.61/0.81 | Adv2 loss/acc 0.62/0.80 | Cos Sim -0.08
> Epoch: 13 | Clf loss/acc 0.60/0.72 | Adv1 loss/acc 0.61/0.81 | Adv2 loss/acc 0.61/0.79 | Cos Sim -0.06
> Epoch: 14 | Clf loss/acc 0.59/0.72 | Adv1 loss/acc 0.62/0.79 | Adv2 loss/acc 0.59/0.79 | Cos Sim -0.04
> Epoch: 15 | Clf loss/acc 0.59/0.72 | Adv1 loss/acc 0.62/0.78 | Adv2 loss/acc 0.58/0.78 | Cos Sim -0.02
> Epoch: 16 | Clf loss/acc 0.59/0.73 | Adv1 loss/acc 0.63/

### EqOpp

In [10]:
fairdef = "EqOpp"

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test = train_test_split(
        x, y, a1, a2, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a1_train, a2_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a1_test, a2_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    opt = Adam(learning_rate=learning_rate)

    model = ZhangMultAdv(xdim=xdim, ydim=ydim, a1dim=a1dim, a2dim=a2dim, batch_size=batch_size, fairdef=fairdef)
    
    ret, dULa1, dULa2, cos_sim = zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A1, A2, Y_hat, A1_hat, A2_hat = fair_evaluation(model, test_data)
    
    clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix = compute_predictive_metrics(Y, Y_hat)
    
    adv1_acc = compute_adv_metrics(A1, A1_hat)
    adv2_acc = compute_adv_metrics(A2, A2_hat)
    
    a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1 = compute_fair_metrics(Y, A1, Y_hat, a1dim)
    a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1 = compute_fair_metrics(Y, A2, Y_hat, a2dim)

    wc_spd, wc_aod, wc_eod = compute_intersectional_fair_metrics(Y, A1, A2, Y_hat, a1dim, a2dim)


    # fair_metrics = (dp, deqodds, deqopp)
    # tradeoff = []
    # for fair_metric in fair_metrics:
    #     tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    # result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    result = ['MultAdvBin4EqOpp', cv_seed]
    result += [clas_acc, clas_f1_micro, clas_f1_macro]
    result += [a1_dp, a1_deqodds, a1_deqopp] + a1_metrics_g0 + a1_metrics_g1 
    result += [a2_dp, a2_deqodds, a2_deqopp] + a2_metrics_g0 + a2_metrics_g1
    result += [wc_spd, wc_aod, wc_eod]
    result += [cos_sim]


    results.append(result)

    del(opt, x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test, train_data, test_data, model, ret)
    del(Y, A1, A2, Y_hat, A1_hat, A2_hat)
    del(clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix, adv1_acc, adv2_acc)
    del(a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1, a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1)
    del(wc_spd, wc_aod, wc_eod)
    del(cos_sim)

> Epoch: 1 | Clf loss/acc 0.83/0.28 | Adv1 loss/acc 0.18/0.81 | Adv2 loss/acc 0.32/0.34 | Cos Sim -0.20
> Epoch: 2 | Clf loss/acc 0.74/0.33 | Adv1 loss/acc 0.18/0.81 | Adv2 loss/acc 0.30/0.34 | Cos Sim -0.21
> Epoch: 3 | Clf loss/acc 0.68/0.49 | Adv1 loss/acc 0.19/0.81 | Adv2 loss/acc 0.28/0.34 | Cos Sim -0.22
> Epoch: 4 | Clf loss/acc 0.65/0.58 | Adv1 loss/acc 0.19/0.81 | Adv2 loss/acc 0.27/0.34 | Cos Sim -0.23
> Epoch: 5 | Clf loss/acc 0.63/0.63 | Adv1 loss/acc 0.19/0.81 | Adv2 loss/acc 0.25/0.34 | Cos Sim -0.24
> Epoch: 6 | Clf loss/acc 0.62/0.68 | Adv1 loss/acc 0.19/0.81 | Adv2 loss/acc 0.24/0.35 | Cos Sim -0.23
> Epoch: 7 | Clf loss/acc 0.61/0.70 | Adv1 loss/acc 0.20/0.81 | Adv2 loss/acc 0.22/0.38 | Cos Sim -0.22
> Epoch: 8 | Clf loss/acc 0.60/0.71 | Adv1 loss/acc 0.20/0.81 | Adv2 loss/acc 0.21/0.45 | Cos Sim -0.21
> Epoch: 9 | Clf loss/acc 0.60/0.71 | Adv1 loss/acc 0.20/0.81 | Adv2 loss/acc 0.20/0.56 | Cos Sim -0.19
> Epoch: 10 | Clf loss/acc 0.59/0.72 | Adv1 loss/acc 0.20/0.81 |

2026-05-15 13:29:19.384587: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 14 | Clf loss/acc 0.59/0.74 | Adv1 loss/acc 0.22/0.67 | Adv2 loss/acc 0.17/0.73 | Cos Sim -0.06
> Epoch: 15 | Clf loss/acc 0.59/0.74 | Adv1 loss/acc 0.22/0.64 | Adv2 loss/acc 0.16/0.72 | Cos Sim -0.03
> Epoch: 16 | Clf loss/acc 0.59/0.74 | Adv1 loss/acc 0.22/0.62 | Adv2 loss/acc 0.16/0.72 | Cos Sim 0.00
> Epoch: 17 | Clf loss/acc 0.59/0.75 | Adv1 loss/acc 0.22/0.60 | Adv2 loss/acc 0.16/0.71 | Cos Sim 0.03
> Epoch: 18 | Clf loss/acc 0.59/0.75 | Adv1 loss/acc 0.23/0.58 | Adv2 loss/acc 0.16/0.70 | Cos Sim 0.06
> Epoch: 19 | Clf loss/acc 0.59/0.75 | Adv1 loss/acc 0.23/0.56 | Adv2 loss/acc 0.16/0.69 | Cos Sim 0.08
> Epoch: 20 | Clf loss/acc 0.59/0.75 | Adv1 loss/acc 0.23/0.55 | Adv2 loss/acc 0.16/0.68 | Cos Sim 0.10
> Epoch: 21 | Clf loss/acc 0.59/0.75 | Adv1 loss/acc 0.24/0.53 | Adv2 loss/acc 0.16/0.68 | Cos Sim 0.11
> Epoch: 22 | Clf loss/acc 0.59/0.75 | Adv1 loss/acc 0.24/0.52 | Adv2 loss/acc 0.16/0.68 | Cos Sim 0.12
> Epoch: 23 | Clf loss/acc 0.59/0.76 | Adv1 loss/acc 0.25/0.50

## Saving into DF then CSV

In [11]:
result_df = pd.DataFrame(results, columns=header)
result_df

,model_name,cv_seed,clas_acc,f1-micro,f1-macro,a1_dp,a1_deqodds,a1_deqopp,a1_TN_g0,a1_FP_g0,...,a2_FN_g0,a2_TP_g0,a2_TN_g1,a2_FP_g1,a2_FN_g1,a2_TP_g1,wc_spd,wc_aod,wc_eod,last_cosine_similarity
0,MultAdvBin4DP,13,0.731061,0.731061,0.644095,0.855144,0.820370,0.809465,1042.0,144.0,...,85.0,25.0,725.0,208.0,236.0,225.0,0.593879,0.554615,0.463629,0.252412
1,MultAdvBin4DP,29,0.719697,0.719697,0.601566,0.845622,0.823114,0.807970,1107.0,117.0,...,81.0,25.0,777.0,146.0,315.0,160.0,0.706520,0.715038,0.693739,0.173155
2,MultAdvBin4DP,42,0.718750,0.718750,0.597255,0.845517,0.793687,0.745514,1103.0,119.0,...,86.0,33.0,769.0,167.0,308.0,146.0,0.685704,0.678015,0.661217,0.267447
3,MultAdvBin4DP,55,0.716856,0.716856,0.619576,0.940015,0.912641,0.946204,1042.0,142.0,...,88.0,29.0,708.0,190.0,289.0,194.0,0.725522,0.794733,0.799177,0.126534
4,MultAdvBin4DP,73,0.721117,0.721117,0.624090,0.913785,0.891943,0.892034,1056.0,179.0,...,84.0,26.0,719.0,235.0,244.0,199.0,0.644110,0.638713,0.612983,0.148858
5,MultAdvBin4EqOdds,13,0.697917,0.697917,0.628677,0.750359,0.731991,0.748850,996.0,190.0,...,83.0,27.0,636.0,297.0,207.0,254.0,0.408027,0.392967,0.346170,0.256505
6,MultAdvBin4EqOdds,29,0.700284,0.700284,0.622028,0.726839,0.703764,0.695327,1046.0,178.0,...,86.0,20.0,656.0,267.0,236.0,239.0,0.398402,0.392220,0.350805,0.233606
7,MultAdvBin4EqOdds,42,0.677557,0.677557,0.602246,0.687162,0.662925,0.667616,1019.0,203.0,...,88.0,31.0,611.0,325.0,229.0,225.0,0.322322,0.329853,0.337979,0.555189
8,MultAdvBin4EqOdds,55,0.694129,0.694129,0.618169,0.806917,0.809993,0.858203,1004.0,180.0,...,90.0,27.0,635.0,263.0,248.0,235.0,0.517508,0.528634,0.536819,0.113469
9,MultAdvBin4EqOdds,73,0.675663,0.675663,0.597020,0.693318,0.675322,0.678832,1019.0,216.0,...,90.0,20.0,611.0,343.0,216.0,227.0,0.284111,0.260037,0.210475,0.135457


In [12]:
result_df.to_csv(f'../../results/{data_name}-mult_adv-{epochs}.csv')